# ClinicalBERT Fine-Tuning on DDX

This notebook is for **supervised diagnosis classification** using `emilyalsentzer/Bio_ClinicalBERT`.

It is separate from the existing retrieval/RAG notebook on purpose:
- Existing notebook: preprocessing + embeddings + FAISS + RAG
- This notebook: fine-tuning + validation + test + confusion matrix


## What this notebook expects

CSV files on Google Drive with at least these columns:
- `combined_text`
- `pathology`

Recommended files:
- `train_processed.csv`
- `validate_processed.csv`
- `test_processed.csv`


In [ ]:
!pip install -q transformers torch pandas scikit-learn tqdm

import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this base path if needed
DRIVE_BASE = Path('/content/drive/MyDrive/DDX')
PROCESSED_DIR = DRIVE_BASE / 'processed'
OUTPUT_DIR = DRIVE_BASE / 'clinicalbert_finetuned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DIR / 'train_processed.csv'
VAL_CSV = PROCESSED_DIR / 'validate_processed.csv'
TEST_CSV = PROCESSED_DIR / 'test_processed.csv'

print('Train:', TRAIN_CSV)
print('Val:', VAL_CSV)
print('Test:', TEST_CSV)


In [ ]:
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
TEXT_COLUMN = 'combined_text'
LABEL_COLUMN = 'pathology'
MAX_LENGTH = 256
BATCH_SIZE = 8
MAX_EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.001

# Recommended run modes:
# - 'smoke' : pipeline check only
# - 'mid'   : best default on Colab for first useful model
# - 'large' : stronger run if session/runtime allows
# - 'full'  : use all processed data
RUN_MODE = 'mid'

RUN_CONFIGS = {
    'smoke': {'train': 200, 'val': 80, 'test': 80, 'max_epochs': 1, 'patience': 1},
    'mid':   {'train': 12000, 'val': 2400, 'test': 2400, 'max_epochs': 4, 'patience': 2},
    'large': {'train': 48000, 'val': 9600, 'test': 9600, 'max_epochs': 6, 'patience': 2},
    'full':  {'train': None, 'val': None, 'test': None, 'max_epochs': 6, 'patience': 2},
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

def stratified_cap_sample(df, label_col, max_samples, seed):
    if max_samples is None or len(df) <= max_samples:
        return df.copy()

    groups = list(df.groupby(label_col, group_keys=False))
    num_classes = max(1, len(groups))
    base_quota = max(1, max_samples // num_classes)

    parts = []
    used_indices = set()
    for _, group in groups:
        take = min(len(group), base_quota)
        sampled_group = group.sample(n=take, random_state=seed)
        parts.append(sampled_group)
        used_indices.update(sampled_group.index.tolist())

    sampled = pd.concat(parts).drop_duplicates()
    remaining_needed = max(0, max_samples - len(sampled))
    if remaining_needed > 0:
        remaining = df.drop(list(used_indices), errors='ignore')
        if len(remaining) > 0:
            extra = remaining.sample(n=min(remaining_needed, len(remaining)), random_state=seed)
            sampled = pd.concat([sampled, extra]).drop_duplicates()

    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

cfg = RUN_CONFIGS[RUN_MODE]
print(f'Running in {RUN_MODE.upper()} mode')
train_df = stratified_cap_sample(train_df, LABEL_COLUMN, cfg['train'], SEED)
active_labels = set(train_df[LABEL_COLUMN].unique())
val_df = val_df[val_df[LABEL_COLUMN].isin(active_labels)].copy()
test_df = test_df[test_df[LABEL_COLUMN].isin(active_labels)].copy()
val_df = stratified_cap_sample(val_df, LABEL_COLUMN, cfg['val'], SEED + 1)
test_df = stratified_cap_sample(test_df, LABEL_COLUMN, cfg['test'], SEED + 2)
MAX_EPOCHS = cfg['max_epochs']
EARLY_STOPPING_PATIENCE = cfg['patience']

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
        raise ValueError(f'{name} is missing required columns')
    df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()
    df = df[df[TEXT_COLUMN] != '']

labels = sorted(train_df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

unseen_val = sorted(set(val_df[LABEL_COLUMN]) - set(labels))
unseen_test = sorted(set(test_df[LABEL_COLUMN]) - set(labels))
if unseen_val:
    raise ValueError(f'Validation has unseen labels: {unseen_val}')
if unseen_test:
    raise ValueError(f'Test has unseen labels: {unseen_test}')

print('Train size:', len(train_df))
print('Val size:', len(val_df))
print('Test size:', len(test_df))
print('Num labels:', len(labels))
print('Max epochs:', MAX_EPOCHS)
print('Early stopping patience:', EARLY_STOPPING_PATIENCE)
train_df[[TEXT_COLUMN, LABEL_COLUMN]].head()


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DiagnosisDataset(Dataset):
    def __init__(self, df, tokenizer, text_col, label_col, label2id, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.label_col = label_col
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_col])
        label = self.label2id[str(row[self.label_col])]
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        item['raw_text'] = text
        return item

def collate_fn(batch):
    out = {
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch]),
        'raw_text': [x['raw_text'] for x in batch],
    }
    if 'token_type_ids' in batch[0]:
        out['token_type_ids'] = torch.stack([x['token_type_ids'] for x in batch])
    return out

train_ds = DiagnosisDataset(train_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, label2id, MAX_LENGTH)
val_ds = DiagnosisDataset(val_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, label2id, MAX_LENGTH)
test_ds = DiagnosisDataset(test_df, tokenizer, TEXT_COLUMN, LABEL_COLUMN, label2id, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
).to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(total_steps * 0.1)),
    num_training_steps=total_steps,
)

def move_batch(batch):
    moved = {}
    for k, v in batch.items():
        moved[k] = v.to(device) if isinstance(v, torch.Tensor) else v
    return moved

def evaluate(model, dataloader):
    model.eval()
    losses = []
    all_true = []
    all_pred = []
    pred_rows = []

    with torch.no_grad():
        for batch in tqdm(dataloader, leave=False):
            batch = move_batch(batch)
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=batch['labels'],
                token_type_ids=batch.get('token_type_ids')
            )
            losses.append(outputs.loss.item())
            preds = torch.argmax(outputs.logits, dim=1)

            true_ids = batch['labels'].detach().cpu().tolist()
            pred_ids = preds.detach().cpu().tolist()
            all_true.extend(true_ids)
            all_pred.extend(pred_ids)

            for text, t, p in zip(batch['raw_text'], true_ids, pred_ids):
                pred_rows.append({
                    'text': text,
                    'true_label': id2label[t],
                    'predicted_label': id2label[p],
                    'correct': t == p,
                })

    avg_loss = float(np.mean(losses)) if losses else 0.0
    acc = accuracy_score(all_true, all_pred)
    report = classification_report(
        all_true,
        all_pred,
        labels=list(range(len(labels))),
        target_names=[id2label[i] for i in range(len(labels))],
        output_dict=True,
        zero_division=0,
    )
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'y_true': all_true,
        'y_pred': all_pred,
        'report': report,
        'prediction_rows': pred_rows,
    }

history = []
best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{MAX_EPOCHS}'):
        batch = move_batch(batch)
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels'],
            token_type_ids=batch.get('token_type_ids')
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_losses.append(loss.item())

    val_metrics = evaluate(model, val_loader)
    record = {
        'epoch': epoch,
        'train_loss': float(np.mean(train_losses)),
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['report']['macro avg']['f1-score'],
    }
    history.append(record)
    print(record)

    if record['val_macro_f1'] > best_val_f1 + EARLY_STOPPING_MIN_DELTA:
        best_val_f1 = record['val_macro_f1']
        best_epoch = epoch
        epochs_without_improvement = 0
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        with open(OUTPUT_DIR / 'label_map.json', 'w', encoding='utf-8') as f:
            json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2, ensure_ascii=False)
    else:
        epochs_without_improvement += 1
        print(f'No val_macro_f1 improvement for {epochs_without_improvement} epoch(s).')
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}. Best epoch was {best_epoch}.')
            break

with open(OUTPUT_DIR / 'training_history.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

print('Best val macro F1:', best_val_f1)
print('Best epoch:', best_epoch)


In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
test_metrics = evaluate(best_model, test_loader)

cm = confusion_matrix(
    test_metrics['y_true'],
    test_metrics['y_pred'],
    labels=list(range(len(labels)))
)
cm_df = pd.DataFrame(cm, index=[id2label[i] for i in range(len(labels))], columns=[id2label[i] for i in range(len(labels))])
report_df = pd.DataFrame(test_metrics['report']).transpose()
pred_df = pd.DataFrame(test_metrics['prediction_rows'])

cm_df.to_csv(OUTPUT_DIR / 'test_confusion_matrix.csv')
report_df.to_csv(OUTPUT_DIR / 'test_classification_report.csv')
pred_df.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

summary = {
    'model_name': MODEL_NAME,
    'run_mode': RUN_MODE,
    'train_size': int(len(train_df)),
    'val_size': int(len(val_df)),
    'test_size': int(len(test_df)),
    'num_labels': int(len(labels)),
    'max_epochs': int(MAX_EPOCHS),
    'best_epoch': int(best_epoch),
    'early_stopping_patience': int(EARLY_STOPPING_PATIENCE),
    'test_loss': float(test_metrics['loss']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_macro_precision': float(test_metrics['report']['macro avg']['precision']),
    'test_macro_recall': float(test_metrics['report']['macro avg']['recall']),
    'test_macro_f1': float(test_metrics['report']['macro avg']['f1-score']),
    'test_weighted_precision': float(test_metrics['report']['weighted avg']['precision']),
    'test_weighted_recall': float(test_metrics['report']['weighted avg']['recall']),
    'test_weighted_f1': float(test_metrics['report']['weighted avg']['f1-score'])
}

with open(OUTPUT_DIR / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
report_df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18, 14))
sns.heatmap(cm_df, cmap='Blues')
plt.title('ClinicalBERT Test Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()
